In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, x_r, theta_0):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [4]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset, base_model: NN, X_train):
    alpha = params['alpha']
    lamb = params['lamb']
    recourse_i = params['recourse_index']
    f_name = f'../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]

        # LIME approximation of original NN
        np.random.seed(i)
        weights_0, bias_0 = lime_explanation(base_model.predict, X_train, x_0)
        weights_0, bias_0 = np.round(weights_0, 4), np.round(bias_0, 4)
        theta_0 = np.hstack((weights_0, bias_0))
        
        # Initalize recourse methods with theta_0
        recourse.set_weights(weights_0)
        recourse.set_bias(bias_0)

        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, x_r, theta_0)

    df_results = pd.DataFrame(results)
    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    
    return df_results

In [5]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = NN(X_train.shape[1])
        base_model.train(X_train.values, y_train.values)
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=None, bias=None, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            if params['append_results']:
                f_name = f"../results/recourse/nn_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
                else:
                    print(f"{f_name} does not exist. The recourses will be created in a new file")

                if recourse_needed_X_test_idx.size == 0:
                    print(f"{f_name} already has all the recourses. Skipping")
                    continue
            
            if params['subsample']:
                rng = np.random.default_rng(seed=seed)
                size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
                if recourse_needed_X_test_idx.shape[0] < size_N:
                    size_N = recourse_needed_X_test_idx.shape[0]
                recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)
            
            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset, base_model, X_train)
            results.append(df_results)

In [6]:
np.hstack((np.array([0.00001, 0.0001]), np.arange(0.001, 0.0105, 0.001), np.arange(0.02, 0.105, 0.01), np.arange(0.1, 1.05, 0.1))).round(7)

array([1.e-05, 1.e-04, 1.e-03, 2.e-03, 3.e-03, 4.e-03, 5.e-03, 6.e-03,
       7.e-03, 8.e-03, 9.e-03, 1.e-02, 2.e-02, 3.e-02, 4.e-02, 5.e-02,
       6.e-02, 7.e-02, 8.e-02, 9.e-02, 1.e-01, 1.e-01, 2.e-01, 3.e-01,
       4.e-01, 5.e-01, 6.e-01, 7.e-01, 8.e-01, 9.e-01, 1.e+00])

In [9]:
alphas = [0.1] # <------------------------
lambdas = [3.0, 0.7, 0.3, 0.1, 0.05, 0.01, 0.001] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:
        
        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = True
        params['subsample'] = False
        params['subsample_size'] = 0.25

        datasets = [GermanDataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running german data...


[L1PSD] [alpha=0.1] [lambda=3.0]: 100%|██████████| 10/10 [34:22<00:00, 206.26s/it]


[L1PSD] Saving results for german run 0


[L1PSD] [alpha=0.1] [lambda=3.0]: 100%|██████████| 5/5 [16:21<00:00, 196.27s/it]


[L1PSD] Saving results for german run 1


...

[L1PSD] Saving results for german run 2


[L1PSD] [alpha=0.1] [lambda=0.001]: 100%|██████████| 6/6 [15:18<00:00, 153.13s/it]


[L1PSD] Saving results for german run 3


[L1PSD] [alpha=0.1] [lambda=0.001]: 100%|██████████| 8/8 [21:28<00:00, 161.04s/it]

[L1PSD] Saving results for german run 4
Finished german



0 [24 19 31] [0 2 1]
1 [17 16 27] [29 34  4]
2 [ 9  4 30] [29  3 17]
3 [ 2  6 27] [30 21 26]
4 [26 33 34] [35  2 36]